# COMP842 Exercise 2 - Proof of Work: Mining, Difficulty and Probability

In [ ]:
# Import SHA-256, timing, and table-processing libraries.
import hashlib
import time
import pandas as pd

def mine_block(block_data, difficulty):
    # A valid hash must begin with the required number of zeros.
    target_prefix = "0" * difficulty
    nonce = 0
    attempts = 0
    start = time.perf_counter()

    # Try different nonces until a valid hash is found.
    while True:
        candidate = f"{block_data}|{nonce}"
        block_hash = hashlib.sha256(candidate.encode("utf-8")).hexdigest()
        attempts += 1

        if block_hash.startswith(target_prefix):
            elapsed = time.perf_counter() - start
            return nonce, block_hash, attempts, elapsed

        nonce += 1

# Test four difficulty levels with 30 blocks each.
difficulties = [2, 3, 4, 5]
runs_per_difficulty = 30
records = []

for difficulty in difficulties:
    print(f"\n{'='*25} DIFFICULTY {difficulty} {'='*25}")
    for run in range(1, runs_per_difficulty + 1):
        # Use unique data so each mining run has a new search problem.
        block_data = (
            f"COMP842|difficulty={difficulty}|run={run}|"
            f"unique={time.time_ns()}"
        )

        nonce, valid_hash, attempts, elapsed = mine_block(
            block_data, difficulty
        )

        # Store the measurements needed for the performance analysis.
        records.append({
            "Difficulty": difficulty,
            "Run": run,
            "Nonce": nonce,
            "Hash Attempts": attempts,
            "Mining Time (s)": elapsed,
            "Valid Hash": valid_hash,
        })

        print(
            f"Run {run:02d} | nonce={nonce:<10} | "
            f"attempts={attempts:<10} | time={elapsed:.6f}s | "
            f"hash={valid_hash}"
        )

# Convert all 120 mining runs into a DataFrame.
runs_df = pd.DataFrame(records)



========================= DIFFICULTY 2 =========================
Run 01 | nonce=311        | attempts=312        | time=0.000304s | hash=00d8c5250f55d0b165774f4ad0d4a918076a26a49f619a124c39880080a4dd8a
Run 02 | nonce=142        | attempts=143        | time=0.000110s | hash=0046de221cc44cceb812313427ed8eda1138552d881612822a6b6cc0cda290d8
Run 03 | nonce=304        | attempts=305        | time=0.000234s | hash=00c5a16e80560cf55aadef03c622fe1703c39b6b692182ef9448cc3613dd9d13
Run 04 | nonce=126        | attempts=127        | time=0.000083s | hash=00ccd6248dc214b4f29d8ed8485f27d41888029e2b2e395ecc9d56bf7bf42b97
Run 05 | nonce=538        | attempts=539        | time=0.000372s | hash=0075946466c2e99c456c6b43ec9e5504b242f9c6f6e4a2e3a03a3c017253da1e
Run 06 | nonce=84         | attempts=85         | time=0.000057s | hash=002f2788bf0e80a74783be89d89acbae93ab85103f067d00f8a3e66947c80815
Run 07 | nonce=138        | attempts=139        | time=0.000098s | hash=00484df3748d730dccc208540040580afe80adda

In [6]:
# Calculate the required performance metrics for each difficulty.
summary_rows = []

for difficulty in difficulties:
    subset = runs_df[runs_df["Difficulty"] == difficulty]

    measured_average_attempts = subset["Hash Attempts"].mean()
    theoretical_attempts = 16 ** difficulty
    total_attempts = subset["Hash Attempts"].sum()
    total_time = subset["Mining Time (s)"].sum()

    summary_rows.append({
        "Difficulty": difficulty,
        "Blocks Mined": len(subset),
        "Average Mining Time (s)": subset["Mining Time (s)"].mean(),
        "Minimum Mining Time (s)": subset["Mining Time (s)"].min(),
        "Maximum Mining Time (s)": subset["Mining Time (s)"].max(),
        "Std Dev Mining Time (s)": subset["Mining Time (s)"].std(ddof=1),
        "Mining Throughput (hashes/s)": total_attempts / total_time,
        "Measured Avg Attempts": measured_average_attempts,
        "Theoretical Attempts (16^d)": theoretical_attempts,
        "Measured/Theoretical": measured_average_attempts / theoretical_attempts,
        "Example Valid Hash": subset.iloc[0]["Valid Hash"],
    })

# Present the summary table required by the question.
summary_df = pd.DataFrame(summary_rows)

print("\n=== REQUIRED PERFORMANCE SUMMARY ===")
print(summary_df.to_string(index=False))



=== REQUIRED PERFORMANCE SUMMARY ===
 Difficulty  Blocks Mined  Average Mining Time (s)  Minimum Mining Time (s)  Maximum Mining Time (s)  Std Dev Mining Time (s)  Mining Throughput (hashes/s)  Measured Avg Attempts  Theoretical Attempts (16^d)  Measured/Theoretical                                               Example Valid Hash
          2            30                 0.000138             9.580035e-07                 0.000474                 0.000121                  1.435239e+06             197.566667                          256              0.771745 00d8c5250f55d0b165774f4ad0d4a918076a26a49f619a124c39880080a4dd8a
          3            30                 0.001637             1.016700e-05                 0.004679                 0.001220                  1.928020e+06            3155.400000                         4096              0.770361 0003d399fe622ff0b6d3611239d0c9345f8e6d8eb97480a392635bcd3ac0da9e
          4            30                 0.034287             2.078916e-03  

In [7]:
# Compare measured growth with the theoretical 16x increase per zero.
growth_rows = []
for previous_difficulty, current_difficulty in zip(difficulties[:-1], difficulties[1:]):
    previous = summary_df[summary_df["Difficulty"] == previous_difficulty].iloc[0]
    current = summary_df[summary_df["Difficulty"] == current_difficulty].iloc[0]

    growth_rows.append({
        "Difficulty Step": f"{previous_difficulty} -> {current_difficulty}",
        "Average Time Ratio": (
            current["Average Mining Time (s)"] /
            previous["Average Mining Time (s)"]
        ),
        "Average Attempts Ratio": (
            current["Measured Avg Attempts"] /
            previous["Measured Avg Attempts"]
        ),
        "Theoretical Ratio": 16.0,
    })

growth_df = pd.DataFrame(growth_rows)
print("\n=== GROWTH BETWEEN DIFFICULTY LEVELS ===")
print(growth_df.to_string(index=False))

# Confirm every result satisfies its requested difficulty.
runs_df["Hash Valid"] = runs_df.apply(
    lambda row: row["Valid Hash"].startswith("0" * int(row["Difficulty"])),
    axis=1
)
print("\nAll 120 hashes satisfy their difficulty:", runs_df["Hash Valid"].all())



=== GROWTH BETWEEN DIFFICULTY LEVELS ===
Difficulty Step  Average Time Ratio  Average Attempts Ratio  Theoretical Ratio
         2 -> 3           11.889226               15.971318               16.0
         3 -> 4           20.950215               28.747470               16.0
         4 -> 5            9.190599                8.894947               16.0

All 120 hashes satisfy their difficulty: True


## Reflection Questions

### 1. Relationship between difficulty and computational effort

My results show that increasing the difficulty requires much more computational effort. The measured average number of attempts increased from **197.57** at difficulty 2 to **3,155.40** at difficulty 3, **90,709.77** at difficulty 4, and **806,858.57** at difficulty 5. The average mining time also increased from **0.000138 seconds** to **0.315119 seconds**.

All 120 hashes were valid, so the extra work was being used to find hashes with the correct number of leading zeros. The measured results are not perfectly regular because the success of each hash attempt is random, but the overall increase matches the expected relationship of approximately `16^difficulty` attempts.

### 2. Is the increase linear or exponential?

The increase is approximately exponential, not linear. The theoretical number of attempts increases by a factor of 16 for every extra leading zero. In my results, the measured attempt ratios were **15.97** from difficulty 2 to 3, **28.75** from 3 to 4, and **8.89** from 4 to 5. These ratios are not all exactly 16 because only 30 blocks were tested at each level and mining is random.

The average mining time shows the same general pattern: it increased by factors of **11.89**, **20.95**, and **9.19** between the difficulty levels. The large standard deviations, especially at difficulty 5, also show that some blocks required far more attempts than others. For example, difficulty 5 had an average time of **0.315119 seconds**, with a maximum of **1.049380 seconds**. Overall, the results support exponential growth rather than a steady linear increase.

### 3. Limitation of Proof of Work and an alternative

The main limitation I observed with Proof of Work is that it uses increasing amounts of computing power and electricity as the difficulty rises. In this experiment, the average attempts increased to more than **806,000** at difficulty 5, even though only one valid hash was finally needed. Most of those calculations were discarded.

Proof of Stake tries to solve this problem by choosing validators based on the cryptocurrency they lock as a stake instead of making them compete with constant hashing. This greatly reduces energy consumption. However, Proof of Stake has its own issues, such as the possibility that users with larger stakes may have more influence over the network.
